# Proteome exploration with ESMC embeddings

A **single** scanpy graph drives everything: a KNN graph on the ESMC embeddings, one UMAP layout, and Leiden clusters — all from the same neighbors graph, with a fixed seed for reproducibility. SAE features are then tested for enrichment per cluster (proteins as "cells", SAE features as "genes").

**Prerequisites:** run the embedding step first, and `pip install -e "..[cluster]"` (scanpy, leidenalg, igraph). Needs `BASEROW_TOKEN` / `BIOHUB_API_TOKEN` in the env.

In [ ]:
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings

cfg = load_config("../config/octopus_chierchiae.yaml")
df = load_embeddings(cfg, prefer_cache=True)   # prefer_cache=False to pull from Baserow
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")
df.head()

## One graph: KNN → UMAP → Leiden

Defaults match the original UMAP (`n_neighbors=15`, `min_dist=0.1`, cosine). Tune **granularity** here: `N_NEIGHBORS`/`MIN_DIST` shape the UMAP, `LEIDEN_RES` sets cluster count (higher = more, finer clusters).

In [ ]:
import scanpy as sc
from och_annotate.analysis import (build_anndata, sae_enrichment, plot_umap,
                                   load_sae_descriptions, annotate_enrichment)

# ---- tunable parameters (defaults reproduce the original UMAP) ----
SEED        = 0
N_NEIGHBORS = 15        # KNN/UMAP granularity (local <-> global structure)
MIN_DIST    = 0.1       # UMAP point spread
METRIC      = "cosine"  # suits language-model embeddings
LEIDEN_RES  = 1.0       # clustering granularity (higher = more clusters)

# Build one AnnData and one neighbors graph; UMAP and Leiden both use it.
adata = build_anndata(df)
sc.pp.neighbors(adata, use_rep="X_esmc", n_neighbors=N_NEIGHBORS, metric=METRIC, random_state=SEED)
sc.tl.umap(adata, min_dist=MIN_DIST, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RES, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)

# Single source of UMAP coords + cluster labels for every plot below.
meta_cols = [c for c in df.columns if c not in ("embedding", "sae_top_features")]
coords = df[meta_cols].copy().reset_index(drop=True)
coords["umap_0"] = adata.obsm["X_umap"][:, 0]
coords["umap_1"] = adata.obsm["X_umap"][:, 1]
coords["leiden"] = adata.obs["leiden"].to_numpy()
print(f"{adata.n_obs} proteins; {coords['leiden'].nunique()} Leiden clusters")

In [ ]:
# Interactive UMAP (scanpy graph) colored by chromosome
plot_umap(coords, color="chromosome", title=f"{cfg.name} — UMAP (chromosome)").show()

In [ ]:
# Same UMAP, colored by Leiden cluster
plot_umap(coords, color="leiden", title=f"{cfg.name} — UMAP (Leiden clusters)").show()

In [ ]:
# Optional: persist a standalone interactive HTML
# from och_annotate.analysis import save_html
# save_html(plot_umap(coords, color="leiden"), "octopus_chierchiae_umap.html")

## SAE-feature enrichment per cluster

Wilcoxon rank-sum on the SAE activation matrix — the marker-gene test, with SAE features in place of genes. Descriptions are merged in when a dictionary file is present.

In [ ]:
# Per-cluster enriched SAE features (FDR in pvals_adj)
enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)

# Attach human-readable descriptions if available (see markdown below to obtain the file).
descriptions = load_sae_descriptions("../data/sae_feature_descriptions.csv")
if descriptions:
    enrich = annotate_enrichment(enrich, descriptions)
    print(f"Annotated with {len(descriptions)} SAE feature descriptions")
else:
    print("No SAE feature-description file found — showing feature indices only.")
enrich.head(30)

In [ ]:
# Top-5 enriched SAE features per cluster (with descriptions when available)
has_desc = "description" in enrich.columns
top5 = (enrich.sort_values(["leiden", "scores"], ascending=[True, False])
              .groupby("leiden", observed=True).head(5))
for cl, g in top5.groupby("leiden", observed=True):
    print(f"\ncluster {cl}:")
    for r in g.itertuples():
        tail = f"  {r.description}" if has_desc else ""
        print(f"  feature {r.sae_feature}  score={r.scores:.1f}{tail}")

# Dotplot of marker SAE features across clusters
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

### SAE feature descriptions — how to enable

Drop a feature-index → description file at `data/sae_feature_descriptions.csv` (columns like `feature,description`; JSON/TSV/parquet also work — columns are auto-detected). The dictionary for `esmc-6b-2024-12-sae-layer60-k64-codebook16384` comes from the **EvolutionaryScale ESMC SAE Atlas** (Hugging Face). Re-run the two cells above and the enrichment gains a `description` column.

### Other next steps
- Write `adata.obs["leiden"]` back to Baserow as a `leiden_cluster` column.
- Tune `LEIDEN_RES` (cluster granularity) and `N_NEIGHBORS` / `MIN_DIST` (UMAP).
- Enrichment sharpens as SAE coverage completes across the proteome.